for each image, rembg mask, optional skin removal (implement in later stage of project), k-means pallete in Lab, compute mean_chroma and neutral_share, save in csv

In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/Users/mariaworkman/VSCode25/fashion-neutrality")
sys.path.insert(0, str(PROJECT_ROOT))

print("using project root:", PROJECT_ROOT)

using project root: /Users/mariaworkman/VSCode25/fashion-neutrality


In [16]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/Users/mariaworkman/VSCode25/fashion-neutrality")

IN_CSV = PROJECT_ROOT / "data/samples/sample_2010_2023_100peryear.csv"
IMG_DIR = PROJECT_ROOT / "data/images/sample_2010_2023_100peryear"

df = pd.read_csv(IN_CSV)
df["local_path"] = df["filename"].apply(lambda f: str(IMG_DIR / f))
df["download_ok"] = df["local_path"].apply(lambda p: Path(p).exists())
df = df[df["download_ok"]].copy()

print("images ready:", len(df))
df.head()

images ready: 1398


,key,designer,season,year,category,city,section,tags,aesthetic,url,filename,local_path,download_ok
0,851646,Jen Kao,Fall,2010,Ready-to-Wear,NaN,Collection,['Evening Dress' 'Clothing' 'Footwear' 'Appare...,4.58,https://assets.vogue.com/photos/55c651b808298d...,0851646_Jen Kao - Fall 2010 Ready-to-Wear [Col...,/Users/mariaworkman/VSCode25/fashion-neutralit...,True
1,144146,Jean Paul Gaultier,Fall,2010,Menswear,NaN,Collection,['Clothing' 'Footwear' 'Apparel' 'Fashion' 'Ja...,4.63,https://assets.vogue.com/photos/55c650ab08298d...,0144146_Jean Paul Gaultier - Fall 2010 Menswea...,/Users/mariaworkman/VSCode25/fashion-neutralit...,True
2,518215,Aquilano.Rimondi,Fall,2010,Ready-to-Wear,NaN,Collection,['Constance Jablonski' 'Evening Dress' 'Clothi...,4.64,https://assets.vogue.com/photos/55c651bd08298d...,0518215_Aquilano.Rimondi - Fall 2010 Ready-to-...,/Users/mariaworkman/VSCode25/fashion-neutralit...,True
3,1013544,Talbot Runhof,Spring,2010,Ready-to-Wear,NaN,Collection,['Clothing' 'Apparel' 'Fashion' 'Person' 'Runw...,4.77,https://assets.vogue.com/photos/55c651af08298d...,1013544_Talbot Runhof - Spring 2010 Ready-to-W...,/Users/mariaworkman/VSCode25/fashion-neutralit...,True
4,286185,Nicole Farhi,Spring,2010,Ready-to-Wear,NaN,Collection,['Clothing' 'Apparel' 'Female' 'Person' 'Human...,4.47,https://assets.vogue.com/photos/55c651a908298d...,0286185_Nicole Farhi - Spring 2010 Ready-to-We...,/Users/mariaworkman/VSCode25/fashion-neutralit...,True


In [17]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from PIL import Image
from sklearn.cluster import MiniBatchKMeans

from rembg import remove  

from src.image_utils import sample_pixels_from_mask
from src.color_utils import kmeans_palette_lab
from src.metrics import neutrality_metrics

# params
N_CLUSTERS = 8
SAMPLE_N = 20000          # pixels sampled per image (tradeoff: speed vs stability)
RANDOM_STATE = 42

OUT_CSV_REMBG = "../data/results/metrics_2010_2023_100peryear_rembg.csv"
os.makedirs(Path(OUT_CSV_REMBG).parent, exist_ok=True)

In [18]:
import numpy as np
import cv2
import io
from PIL import Image
from rembg import remove, new_session

session = new_session("u2net")

def rembg_mask_binarized(bgr_img, session):
    """
    Returns HxW uint8 mask (0 background, 255 foreground)
    Works whether rembg.remove returns ndarray or PNG bytes.
    """
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)

    out = remove(rgb, session=session)

    # Case 1: rembg returns ndarray
    if isinstance(out, np.ndarray):
        arr = out
        # could be RGB or RGBA
        if arr.ndim == 3 and arr.shape[2] == 4:
            alpha = arr[:, :, 3]
            return (alpha > 0).astype(np.uint8) * 255
        else:
            # no alpha channel; treat non-black as foreground
            return ((arr.sum(axis=2) > 0).astype(np.uint8)) * 255

    # Case 2: rembg returns bytes (PNG)
    if isinstance(out, (bytes, bytearray)):
        im = Image.open(io.BytesIO(out)).convert("RGBA")
        alpha = np.array(im)[:, :, 3]
        return (alpha > 0).astype(np.uint8) * 255

    raise TypeError(f"Unexpected rembg output type: {type(out)}")



In [19]:
bgr = cv2.imread(df.iloc[0]["local_path"])
mask = rembg_mask_binarized(bgr, session)
print(mask.shape, mask.dtype, np.unique(mask)[:10])


(1024, 683) uint8 [  0 255]


In [20]:
## placeholder for later!!

def remove_skin_pixels_lab(pixels_lab: np.ndarray) -> np.ndarray:
    """
    placeholder for later. for now, do nothing.
    pixels_lab: Nx3 float (L,a,b)
    """
    return pixels_lab

In [ ]:
N_CLUSTERS = 8
SAMPLE_N = 20000
RANDOM_STATE = 42
TAU = 20.0

results = []

for i, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
    local_path = row["local_path"]

    out = {
        "key": row.get("key"),
        "designer": row.get("designer"),
        "season": row.get("season"),
        "year": row.get("year"),
        "category": row.get("category"),
        "city": row.get("city"),
        "section": row.get("section"),
        "tags": row.get("tags"),
        "aesthetic": row.get("aesthetic"),
        "url": row.get("url"),
        "filename": row.get("filename"),
        "local_path": local_path,
        "ok": False,
        "error": None,
    }

    try:
        bgr = cv2.imread(local_path, cv2.IMREAD_COLOR)
        if bgr is None:
            raise ValueError("cv2.imread returned None")

        mask = rembg_mask_binarized(bgr, session)

        pixels_bgr = sample_pixels_from_mask(
            bgr, mask, n=SAMPLE_N, seed=RANDOM_STATE + i
        )

        if pixels_bgr is None:
            pixels_bgr = sample_pixels_full(
                bgr, n=SAMPLE_N, seed=RANDOM_STATE + i
            )

        centers_lab, weights = kmeans_palette_lab(
            pixels_bgr,
            k=N_CLUSTERS,
            seed=RANDOM_STATE
        )

        mean_chroma, neutral_share = neutrality_metrics(
            centers_lab=centers_lab,
            weights=weights,
            tau=TAU
        )

        out["mean_chroma"] = mean_chroma
        out["neutral_share"] = neutral_share
        out["ok"] = True

    except Exception as e:
        out["error"] = str(e)

    results.append(out)

metrics_df = pd.DataFrame(results)

OUT_CSV = PROJECT_ROOT / "data/results/metrics_full_run.csv"
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(OUT_CSV, index=False)